# P6-2 · Bài 8b · Preference tuning

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q datasets trl

## Lý thuyết

In [ ]:
{"prompt":   [{"role": "user", "content": "Đơn DH2051 giao chậm 3 ngày rồi, shop xử lý sao?"}],
 "chosen":   [{"role": "assistant", "content": "Xin lỗi anh/chị vì đơn DH2051 bị chậm. Em đã kiểm tra: ..."}],
 "rejected": [{"role": "assistant", "content": "Cảm ơn quý khách đã liên hệ. Chúng tôi luôn trân trọng ..."}]}

{'prompt': [{'role': 'user', 'content': 'Đơn DH2051 giao chậm 3 ngày rồi, shop xử lý sao?'}], 'chosen': [{'role': 'assistant', 'content': 'Xin lỗi anh/chị vì đơn DH2051 bị chậm. Em đã kiểm tra: ...'}], 'rejected': [{'role': 'assistant', 'content': 'Cảm ơn quý khách đã liên hệ. Chúng tôi luôn trân trọng ...'}]}


In [ ]:
# Cần GPU, model, LoRA và dataset ở các bước trước: chạy trên Colab
from trl import DPOTrainer, DPOConfig
args = DPOConfig(output_dir="qwen-dpo", beta=0.1, learning_rate=2e-5, num_train_epochs=1)
trainer = DPOTrainer(model=model, ref_model=None, args=args, train_dataset=ds["train"],
                     processing_class=tok, peft_config=lora_cfg)   # ref_model=None + LoRA: tham chiếu = model nền

## Bài tập 5: Chuẩn bị dataset cho DPO {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json
from datasets import Dataset

# file mẫu 3 dòng để chạy thử: một dòng hòa sẽ bị bỏ
demo = [{"prompt": "Đơn giao chậm, shop xử lý sao?", "answer_a": "Xin lỗi anh/chị, em kiểm tra ngay: ...", "answer_b": "Cảm ơn quý khách đã liên hệ ...", "preferred": "a"},
        {"prompt": "Bàn có kèm gương không?", "answer_a": "Có ạ.", "answer_b": "Có, gương 60×40 cm gắn sẵn trên bàn.", "preferred": "b"},
        {"prompt": "Shop mở cửa mấy giờ?", "answer_a": "8h–21h.", "answer_b": "Từ 8 giờ tới 21 giờ.", "preferred": "tie"}]
with open("prefs.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(d, ensure_ascii=False) + "\n" for d in demo)

rows = []
for line in open("prefs.jsonl", encoding="utf-8"):
    r = json.loads(line)
    if r["preferred"] not in ("a", "b"):
        continue
    good, bad = (r["answer_a"], r["answer_b"]) if r["preferred"] == "a" else (r["answer_b"], r["answer_a"])
    rows.append({
        "prompt": [{"role": "user", "content": r["prompt"]}],
        "chosen": [{"role": "assistant", "content": good}],
        "rejected": [{"role": "assistant", "content": bad}],
    })
ds = Dataset.from_list(rows).train_test_split(test_size=0.5, seed=0)    # dữ liệu thật: test_size=0.1
print(ds)
print(ds["train"][0]["chosen"])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1
    })
})
[{'role': 'assistant', 'content': 'Có, gương 60×40 cm gắn sẵn trên bàn.'}]


## Bài tập 6: Tính loss DPO cho một batch {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn.functional as F
pi_chon = torch.tensor([-42.0, -55.0, -30.0, -61.0]); pi_loai = torch.tensor([-48.0, -53.0, -36.0, -60.0])
ref_chon = torch.tensor([-45.0, -54.0, -33.0, -60.0]); ref_loai = torch.tensor([-46.0, -54.0, -34.0, -61.0])

def dpo(beta):
    # TODO: trả về (loss, accuracy, margin trung bình)
    return 0.0, 0.0, 0.0

In [ ]:
def dpo(beta):
    r_chon, r_loai = beta * (pi_chon - ref_chon), beta * (pi_loai - ref_loai)
    m = r_chon - r_loai
    return -F.logsigmoid(m).mean().item(), (m > 0).float().mean().item(), m.mean().item()

for b in (0.1, 0.5):
    l, acc, m = dpo(b)
    print(f"β = {b}: loss {l:.3f}, xếp đúng {acc:.0%}, margin TB {m:.2f}")

β = 0.1: loss 0.636, xếp đúng 50%, margin TB 0.15
β = 0.5: loss 0.696, xếp đúng 50%, margin TB 0.75


## Bài tập 7: Học reward model từ các cặp so sánh {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)
w_that = np.array([2.0, 1.0, 0.4])                  # sở thích thật của người chấm (cả độ dài!)
A, B = rng.normal(size=(800, 3)), rng.normal(size=(800, 3))
p_A = 1 / (1 + np.exp(-(A - B) @ w_that))
A_thang = (rng.random(800) < p_A).astype(int)
# TODO: train trên 600 cặp đầu với đặc trưng A - B; acc trên 200 cặp cuối; w_hoc

In [ ]:
D = A - B
rm = LogisticRegression(fit_intercept=False).fit(D[:600], A_thang[:600])
acc = rm.score(D[600:], A_thang[600:])
w_hoc = rm.coef_[0]
print("trọng số học được [đúng, lịch sự, độ dài]:", w_hoc.round(2), "| thật:", w_that)
print(f"độ chính xác trên cặp kiểm tra: {acc:.2f}")

trọng số học được [đúng, lịch sự, độ dài]: [2.01 0.99 0.38] | thật: [2.  1.  0.4]
độ chính xác trên cặp kiểm tra: 0.87


## Bài tập 8: Kiểm tra dữ liệu ưu tiên trước khi train {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
data = [("Bàn có mấy màu?", "Dạ có 3 màu: trắng, hồng và gỗ tự nhiên ạ.", "3 màu.", "A", "A"),
        ("Giao bao lâu?", "Dạ nội thành 1–2 ngày ạ.", "Dạ nội thành 1–2 ngày ạ.", "A", "B"),
        ("Có trả góp không?", "Dạ có, qua thẻ tín dụng 0% trong 6 tháng ạ.", "Không.", "A", "A"),
        ("Đổi trả thế nào?", "Dạ trong 7 ngày nếu lỗi ạ.", "Dạ shop hỗ trợ đổi trả trong vòng 7 ngày kể từ ngày nhận hàng, áp dụng cho mọi trường hợp ạ.", "A", "A"),
        ("Gương có chống mờ không?", "Dạ có lớp chống mờ hơi nước ạ.", "Dạ có ạ, gương rất đẹp, nhiều người mua, shop uy tín lâu năm.", "A", "B"),
        ("Ship tỉnh không?", "Dạ shop giao toàn quốc ạ, phí từ 30.000đ.", "Có.", "A", "A")]
# TODO: giong, ti_le_dai_hon, dong_y, giu

In [ ]:
giong = sum(c.strip() == r.strip() for _, c, r, *_ in data)
ti_le_dai_hon = sum(len(c) > len(r) for _, c, r, *_ in data) / len(data)
dong_y = sum(a == b for *_, a, b in data) / len(data)
giu = [d for d in data if d[1].strip() != d[2].strip() and d[3] == d[4]]
print(f"giống hệt: {giong}, chosen dài hơn: {ti_le_dai_hon:.0%}, hai người đồng ý: {dong_y:.0%}, giữ {len(giu)}/{len(data)}")

giống hệt: 1, chosen dài hơn: 50%, hai người đồng ý: 67%, giữ 4/6
